# Kenya SME Policy Knowledge Assistant (Advanced RAG + Evaluation)

Small and medium enterprises (SMEs) in Kenya often struggle to interpret:

- Tax compliance rules
- Government grant eligibility
- Digital service regulations

I decided to build a Retrieval-Augmented Generation (RAG) assistant that:

1. Stores policy documents as embeddings.
2. Retrieves the most relevant chunks.
3. Generates grounded answers.
4. Evaluates retrieval quality using MRR.
5. Uses GPT-4 as a judge to score answer relevance.

## Concepts Applied

- Vector embeddings
- Semantic similarity search
- Manual RAG (no LangChain)
- Retrieval evaluation (MRR)
- LLM-as-Judge scoring
- Chunking strategies

In [4]:
pip install openai python-dotenv numpy scikit-learn --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4"
EMBED_MODEL = "text-embedding-3-small"

sample SME Policy docs

In [5]:
documents = [
    "Kenyan SMEs must file VAT returns monthly if turnover exceeds KES 5M annually.",
    "Youth Enterprise Development Fund provides low-interest loans to youth-owned businesses.",
    "Digital Service Tax applies to non-resident digital marketplace providers operating in Kenya.",
    "Businesses must register with KRA for PIN before operating legally."
]

create embeddings

In [6]:
def embed_text(text):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return response.data[0].embedding

doc_embeddings = [embed_text(doc) for doc in documents]

retrieval

In [7]:
def retrieve(query, top_k=2):
    query_embedding = embed_text(query)
    similarities = cosine_similarity(
        [query_embedding],
        doc_embeddings
    )[0]

    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [documents[i] for i in top_indices]

generate RAG answer

In [8]:
def generate_answer(query):
    context = retrieve(query)
    context_text = "\n".join(context)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer based only on provided context."},
            {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

evaluate with MRR

In [9]:
def compute_mrr(query, correct_doc_index):
    retrieved = retrieve(query, top_k=len(documents))
    ranks = [documents.index(doc) for doc in retrieved]

    if correct_doc_index in ranks:
        rank = ranks.index(correct_doc_index) + 1
        return 1 / rank
    return 0

TEST

In [10]:
query = "How often do SMEs file VAT returns in Kenya?"
print(generate_answer(query))
print("MRR:", compute_mrr(query, 0))

SMEs in Kenya must file VAT returns monthly.
MRR: 1.0
